# Full-Stack Machine Learning Assignment: Pharmaceutical Dataset

**Goal:** train one regression model and one classification model using the attached Pakistan pharmaceutical pricing dataset.

- Regression target: `Price_After` (numeric final price in PKR)
- Classification target: clean `Availability` labels (`Available` or `Sold Out`)
- Regression evaluation: R² and mean absolute error
- Classification evaluation: accuracy and classification report
- Save models with `joblib` for use by the FastAPI website

Run every cell from top to bottom. The dataset is expected at `data/pharmaceutical_data.xlsx` beside this notebook. If you open the notebook from the project folder, the path will work.


In [1]:
# If a package is missing, run this in a notebook cell:
# %pip install pandas openpyxl scikit-learn joblib


In [2]:
import os
import re
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score, classification_report

os.makedirs("models", exist_ok=True)


In [3]:
# Load the dataset
df = pd.read_excel("data/pharmaceutical_data.xlsx")

print("Dataset shape:", df.shape)
display(df.head())


Dataset shape: (1630, 7)


,Name,Company,Price_before,Discount,Price_After,Pack_Size,Availability
0,Zestril,ICI Pakistan Limited,Rs 202,10% Off,182.0,1x14's,Available
1,Zestril,ICI Pakistan Limited,Rs 202,10% Off,182.0,1x14's,Available
2,Zestril,ICI Pakistan Limited,Rs 202,10% Off,182.0,1x14's,Available
3,Zestril,ICI Pakistan Limited,Rs 388,10% Off,350.0,1x14's,Available
4,Zestril,ICI Pakistan Limited,Rs 388,10% Off,350.0,1x14's,Available


In [4]:
# Check column types and missing values
df.info()
display(df.isnull().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1630 entries, 0 to 1629
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Name          1627 non-null   object 
 1   Company       1627 non-null   object 
 2   Price_before  1627 non-null   object 
 3   Discount      1491 non-null   object 
 4   Price_After   1627 non-null   float64
 5   Pack_Size     1627 non-null   object 
 6   Availability  1627 non-null   object 
dtypes: float64(1), object(6)
memory usage: 89.3+ KB


Name              3
Company           3
Price_before      3
Discount        139
Price_After       3
Pack_Size         3
Availability      3
dtype: int64

In [5]:
# Check duplicate rows and the values in Availability
print("Duplicate rows:", df.duplicated().sum())
display(df["Availability"].astype(str).str.strip().value_counts().head(20))


Duplicate rows: 664


Availability
Sold Out     503
Available    457
1 Vial        98
1x10's        49
30ml          28
60ml          26
10's          24
1x14's        23
10            22
3x10's        22
2x10's        17
2x7's         16
20's          14
5's           13
30            13
1 x 10's      12
1             12
30's          10
20            10
3 x 10's       7
Name: count, dtype: int64

In [6]:
# Clean the price and discount columns
df["Price_before_numeric"] = pd.to_numeric(
    df["Price_before"].astype(str).str.replace("Rs", "", regex=False).str.replace(",", "", regex=False).str.strip(),
    errors="coerce"
)

df["Discount_percent"] = pd.to_numeric(
    df["Discount"].astype(str).str.extract(r"(\d+(?:\.\d+)?)")[0],
    errors="coerce"
).fillna(0)

df["Price_After"] = pd.to_numeric(df["Price_After"], errors="coerce")

for col in ["Name", "Company", "Pack_Size"]:
    df[col] = df[col].fillna("Unknown").astype(str).str.strip()

# Remove exact duplicate rows
df = df.drop_duplicates().copy()
print("Shape after removing exact duplicates:", df.shape)
display(df[["Name", "Company", "Price_before_numeric", "Discount_percent", "Price_After", "Pack_Size", "Availability"]].head())


Shape after removing exact duplicates: (966, 9)


,Name,Company,Price_before_numeric,Discount_percent,Price_After,Pack_Size,Availability
0,Zestril,ICI Pakistan Limited,202.0,10.0,182.0,1x14's,Available
3,Zestril,ICI Pakistan Limited,388.0,10.0,350.0,1x14's,Available
6,Zestril,ICI Pakistan Limited,748.0,10.0,673.0,1x14's,Available
9,Zofran 8mg,GlaxoSmithKline,650.0,10.0,585.0,1,Available
10,Zeegap,Hilton,376.0,10.0,338.0,1x14's,Available


## Part A — Regression model

The model predicts a medicine's final price (`Price_After`). We use Linear Regression, as required by the assignment. The pipeline fills missing values and converts text categories into numeric features.


In [7]:
reg_features = ["Name", "Company", "Price_before_numeric", "Discount_percent", "Pack_Size"]
reg_data = df[reg_features + ["Price_After"]].dropna(subset=["Price_After"]).copy()

X_reg = reg_data[reg_features]
y_reg = reg_data["Price_After"]

reg_categorical = ["Name", "Company", "Pack_Size"]
reg_numeric = ["Price_before_numeric", "Discount_percent"]

reg_preprocess = ColumnTransformer([
    ("categories", Pipeline([
        ("fill", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore"))
    ]), reg_categorical),
    ("numbers", Pipeline([
        ("fill", SimpleImputer(strategy="median"))
    ]), reg_numeric)
])

reg_model = Pipeline([
    ("preprocess", reg_preprocess),
    ("model", LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

reg_model.fit(X_train, y_train)
reg_predictions = reg_model.predict(X_test)

print("Rows used:", len(reg_data))
print("R² score:", round(r2_score(y_test, reg_predictions), 3))
print("Mean absolute error (PKR):", round(mean_absolute_error(y_test, reg_predictions), 2))

joblib.dump(reg_model, "models/regression_model.pkl")
joblib.dump(reg_features, "models/regression_features.pkl")
print("Regression model saved.")


Rows used: 965
R² score: 1.0
Mean absolute error (PKR): 1.13
Regression model saved.


## Part B — Classification model

The source `Availability` column contains mixed values, such as pack sizes, in addition to `Available` and `Sold Out`. To avoid pretending those other values are valid labels, we keep only the two clear categories. The model predicts one of these two labels.


In [8]:
df["Availability_clean"] = df["Availability"].astype(str).str.strip().str.lower()
valid_labels = {"available": "Available", "sold out": "Sold Out"}
df["Availability_clean"] = df["Availability_clean"].map(valid_labels)

class_features = ["Name", "Company", "Price_before_numeric", "Discount_percent", "Pack_Size", "Price_After"]
class_data = df[class_features + ["Availability_clean"]].dropna(subset=["Availability_clean"]).copy()
class_data = class_data.dropna(subset=["Price_After"])

X_class = class_data[class_features]
y_class = class_data["Availability_clean"]

print("Rows used:", len(class_data))
display(y_class.value_counts())


Rows used: 447


Availability_clean
Sold Out     262
Available    185
Name: count, dtype: int64

In [9]:
class_categorical = ["Name", "Company", "Pack_Size"]
class_numeric = ["Price_before_numeric", "Discount_percent", "Price_After"]

class_preprocess = ColumnTransformer([
    ("categories", Pipeline([
        ("fill", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore"))
    ]), class_categorical),
    ("numbers", Pipeline([
        ("fill", SimpleImputer(strategy="median"))
    ]), class_numeric)
])

class_model = Pipeline([
    ("preprocess", class_preprocess),
    ("model", DecisionTreeClassifier(max_depth=8, random_state=42, class_weight="balanced"))
])

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class, y_class, test_size=0.2, random_state=42, stratify=y_class
)

class_model.fit(X_train_c, y_train_c)
class_predictions = class_model.predict(X_test_c)

print("Accuracy:", round(accuracy_score(y_test_c, class_predictions), 3))
print("\nClassification report:")
print(classification_report(y_test_c, class_predictions, zero_division=0))

joblib.dump(class_model, "models/classification_model.pkl")
joblib.dump(class_features, "models/classification_features.pkl")
print("Classification model saved.")


Accuracy: 0.644

Classification report:
              precision    recall  f1-score   support

   Available       0.56      0.59      0.58        37
    Sold Out       0.71      0.68      0.69        53

    accuracy                           0.64        90
   macro avg       0.63      0.64      0.64        90
weighted avg       0.65      0.64      0.65        90



Classification model saved.


## Part C — Test one prediction

The next cell demonstrates using the saved regression pipeline with one sample input. The web app uses the same saved model files.


In [10]:
sample = pd.DataFrame([{
    "Name": "Zestril",
    "Company": "ICI Pakistan Limited",
    "Price_before_numeric": 202,
    "Discount_percent": 10,
    "Pack_Size": "1x14's"
}])

loaded_regression_model = joblib.load("models/regression_model.pkl")
predicted_price = loaded_regression_model.predict(sample)[0]
print("Predicted final price (PKR):", round(float(predicted_price), 2))


Predicted final price (PKR): 182.05


## Conclusion

- Linear Regression predicts the numeric final medicine price.
- The Decision Tree classifier predicts `Available` or `Sold Out` using only rows with those clear labels.
- R² and mean absolute error summarize regression performance; accuracy and the classification report summarize classification performance.
- The saved model pipelines include preprocessing, so the API can use raw form inputs in the same feature format.

**Important:** These are educational predictions based on a small, imperfect dataset. Do not use them for actual medicine decisions. Run the notebook to obtain your real scores; do not copy guessed scores into the report.
